In [1]:
import random
import numpy as np
from pathlib import Path
from the_well.data import WellDataset

import torch
from torch.utils.data import DataLoader

from modules import *
from losses import *
from datasets import HelmholtzDataset

In [2]:
SEED         = 42
EPOCHS       = 20
BATCH_SIZE   = 8
LR           = 1e-3

DATASET_PATH = "/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase"
OUTPUT_DIR   = "./models"

MODES1       = 16   # Modos de Fourier na primeira dimensão espacial (x)
MODES2       = 16   # Modos de Fourier na segunda dimensão espacial (y)
WIDTH        = 32   # Número de canais internos (largura do modelo)
DEPTH        = 4    # Quantidade de camadas de Fourier
PROJ_DIM     = 128  # Dimensão da MLP de projeção para a saída

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [4]:
train_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="train"
)

validation_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="valid"
)

train_ds = HelmholtzDataset(train_dataset)
val_ds = HelmholtzDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=4,
)

### Teste 1

In [5]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

criterion = relative_l2_loss

/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


In [6]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/20:   0%|          | 0/2548 [00:00<?, ?it/s]/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/torch/_inductor/lowering.py:2212: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
W0622 19:51:55.923000 17941 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
Epoch 1/20: 100%|██████████| 2548/2548 [21:37<00:00,  1.96it/s, loss=0.991098]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `p

Epoch 001 | train = 0.991098 | val = 0.845437 | best_val = 0.845437


Epoch 2/20: 100%|██████████| 2548/2548 [20:14<00:00,  2.10it/s, loss=0.987134]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 002 | train = 0.987134 | val = 0.807976 | best_val = 0.807976


Epoch 3/20: 100%|██████████| 2548/2548 [20:02<00:00,  2.12it/s, loss=0.987013]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


...

Epoch 018 | train = 0.985518 | val = 0.792563 | best_val = 0.758614


Epoch 19/20: 100%|██████████| 2548/2548 [09:56<00:00,  4.27it/s, loss=0.985248]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 019 | train = 0.985248 | val = 0.790036 | best_val = 0.758614


Epoch 20/20: 100%|██████████| 2548/2548 [09:53<00:00,  4.29it/s, loss=0.985060]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 020 | train = 0.985060 | val = 0.788603 | best_val = 0.758614


Resultado teste 1

Epoch 020 | train = 0.985060 | val = 0.788603 | best_val = 0.758614

### Teste 2

In [7]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d_v1(
    modes1=MODES1,
    modes2=MODES2,
    width=64,
    width1= 64,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=192
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

criterion = relative_l2_loss

In [8]:
history2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/20: 100%|██████████| 2548/2548 [18:47<00:00,  2.26it/s, loss=0.994959]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 001 | train = 0.994959 | val = 0.847745 | best_val = 0.847745


Epoch 2/20: 100%|██████████| 2548/2548 [18:38<00:00,  2.28it/s, loss=0.988145]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 002 | train = 0.988145 | val = 0.781171 | best_val = 0.781171


Epoch 3/20: 100%|██████████| 2548/2548 [18:38<00:00,  2.28it/s, loss=0.987868]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


...

Epoch 018 | train = 0.984662 | val = 0.781061 | best_val = 0.781061


Epoch 19/20: 100%|██████████| 2548/2548 [18:38<00:00,  2.28it/s, loss=0.985519]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 019 | train = 0.985519 | val = 0.793971 | best_val = 0.781061


Epoch 20/20: 100%|██████████| 2548/2548 [18:38<00:00,  2.28it/s, loss=0.984769]
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


Epoch 020 | train = 0.984769 | val = 0.789566 | best_val = 0.781061


Resultados Teste 2

Epoch 020 | train = 0.984769 | val = 0.789566 | best_val = 0.781061

### Teste 3

In [9]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d_v1(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    width1= WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

criterion = relative_l2_loss

In [ ]:
history3 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/20:   1%|          | 29/2548 [00:14<10:32,  3.98it/s, loss=1.050622] 